# FLChain, GBSG2 and WHAS500 on 10 data splits

In notebook 09, VaDeSC and the classical models looked about the same on these three datasets, but with only 3 runs we could not tell if the small differences were real. Here we use 10 data splits (splits 0-9, as in notebook 09), and we also try tuning VaDeSC.

What we check:
1. if VaDeSC is better or worse than Cox, RSF, Weibull AFT and DSM,
2. if tuning VaDeSC helps compared with the paper settings for these datasets (2 clusters, latent size 4, Weibull k = 1).

WHAS500 is used without `los`, because it is recorded after admission.

As in notebook 16, the tests use one value per split (the mean over seeds), with the Bonferroni correction for the 5 comparisons per dataset.

Running everything takes about 2 hours on a CPU. Set `QUICK = True` for a short test.

In [1]:
import sys
sys.path.append("..")
from dataclasses import replace

import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed
from scipy import stats
from sksurv.metrics import concordance_index_censored

from src.baselines import run_cox, run_dsm, run_rsf, run_weibull_aft
from src.data import load_sksurv
from src.vadesc import VaDeSC, evaluate_vadesc, tensor, train_vadesc

QUICK = False
SPLITS = [0, 1] if QUICK else list(range(10))
SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = 30 if QUICK else 300
N_JOBS = 6
PAPER = dict(K=2, d_lat=4, weibull_k=1.0)


class VaDeSCLearnK(VaDeSC):
    # same model, but the Weibull shape k is learned (initialised at weibull_k), as in notebook 14
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.learn_k = True


def drop_features(d, names):
    keep = [i for i, f in enumerate(d.features) if f not in names]
    return replace(d, X_tr=d.X_tr[:, keep], X_va=d.X_va[:, keep], X_te=d.X_te[:, keep],
                   features=[d.features[i] for i in keep])


LOAD = {"FLChain": lambda s: load_sksurv("FLChain", seed=s), "GBSG2": lambda s: load_sksurv("GBSG2", seed=s),
        "WHAS500": lambda s: drop_features(load_sksurv("WHAS500", seed=s), ["los"])}
data = {(ds, sp): f(sp) for ds, f in LOAD.items() for sp in SPLITS}


def split_test(per_split, a, b, n_comp=1):
    diff = per_split[a] - per_split[b]
    n = len(diff)
    ci = stats.t.ppf(0.975, n - 1) * diff.std(ddof=1) / np.sqrt(n)
    p = stats.ttest_1samp(diff, 0).pvalue
    return {"comparison": f"{a}  minus  {b}", "mean diff": diff.mean(),
            "95% CI": f"[{diff.mean() - ci:+.3f}, {diff.mean() + ci:+.3f}]",
            "higher in": f"{(diff > 0).sum()}/{n} splits", "p-value": p, "p (Bonferroni)": min(1.0, p * n_comp)}

## 1. Classical models

Cox, RSF, Weibull AFT and DSM, once per split.

In [2]:
def run_classical(ds, sp, d):
    torch.set_num_threads(2)
    out = []
    for model, fn in [("Cox PH", run_cox), ("RSF", lambda x: run_rsf(x, seed=sp)), ("Weibull AFT", run_weibull_aft),
                      ("DSM", lambda x: run_dsm(x, seed=sp))]:
        m = fn(d)
        out.append({"dataset": ds, "split": sp, "method": model, "C-index": m["C-index"], "IBS": m["IBS"]})
    return out


cls = pd.DataFrame([r for rows in Parallel(n_jobs=N_JOBS)(delayed(run_classical)(ds, sp, d) for (ds, sp), d in data.items())
                    for r in rows])
cls.groupby(["dataset", "method"])[["C-index", "IBS"]].agg(["mean", "std"]).round(3)

C-index           IBS       
                       mean    std   mean    std
dataset method                                  
FLChain Cox PH        0.794  0.006  0.108  0.002
        DSM           0.795  0.007  0.108  0.002
        RSF           0.789  0.007  0.109  0.002
        Weibull AFT   0.795  0.006  0.109  0.002
GBSG2   Cox PH        0.645  0.022  0.190  0.005
        DSM           0.658  0.024  0.189  0.007
        RSF           0.677  0.029  0.183  0.008
        Weibull AFT   0.645  0.022  0.191  0.004
WHAS500 Cox PH        0.761  0.031  0.167  0.013
        DSM           0.757  0.024  0.165  0.013
        RSF           0.759  0.023  0.171  0.009
        Weibull AFT   0.761  0.031  0.169  0.014

## 2. VaDeSC with the paper settings and tuned

First VaDeSC with the paper settings (3 seeds per split). Then we try 18 settings: number of clusters (2, 3, 4), latent size (4, 8) and survival distribution (Weibull with k = 1 like the paper, Weibull with a learned k, or log-normal). On every split we keep the setting with the best validation C-index, train it with 3 seeds and test it on the test set.

In [3]:
def run_vadesc(d, seed, ds, sp, label, model_cls=VaDeSC, **cfg):
    torch.set_num_threads(2)
    cfg = {**PAPER, **cfg}
    model = train_vadesc(d, epochs=EPOCHS, seed=seed, model_cls=model_cls, **cfg)
    m, _, _ = evaluate_vadesc(model, d)
    risk_va = model.risk_score(tensor(d.X_va)).cpu().numpy()
    val_c = concordance_index_censored(d.e_va.astype(bool), d.t_va, risk_va)[0]
    k_hat = float(model.shape()) if cfg.get("dist", "weibull") == "weibull" else np.nan
    return {"dataset": ds, "split": sp, "seed": seed, "method": label, "C-index": m["C-index"], "IBS": m["IBS"],
            "val C-index": val_c, "k": k_hat}


GRID = [dict(K=K, d_lat=dl, **dist) for K in ([2] if QUICK else [2, 3, 4]) for dl in [4, 8]
        for dist in [dict(weibull_k=1.0), dict(weibull_k=1.0, model_cls=VaDeSCLearnK), dict(dist="lognormal")]]


def name_of(cfg):
    dist = "log-normal" if cfg.get("dist") == "lognormal" else ("Weibull, learned k" if "model_cls" in cfg else "Weibull, k = 1")
    return f"K={cfg['K']}, d={cfg['d_lat']}, {dist}"


paper = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=2)(
    delayed(run_vadesc)(d, s, ds, sp, "VaDeSC (paper config)") for (ds, sp), d in data.items() for s in SEEDS))
tune = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=2)(
    delayed(run_vadesc)(d, 0, ds, sp, name_of(cfg), **cfg) for (ds, sp), d in data.items() for cfg in GRID))
best = tune.loc[tune.groupby(["dataset", "split"])["val C-index"].idxmax()].set_index(["dataset", "split"])["method"]
cfg_of = {name_of(c): c for c in GRID}
tuned = pd.DataFrame(Parallel(n_jobs=N_JOBS)(
    delayed(run_vadesc)(data[(ds, sp)], s, ds, sp, "VaDeSC (tuned)", **cfg_of[best[(ds, sp)]]) for (ds, sp) in data for s in SEEDS))
tuned["setting"] = [best[(ds, sp)] for ds, sp in zip(tuned.dataset, tuned.split)]
pd.concat([cls, paper, tune.assign(stage="tuning"), tuned]).to_csv("../results/real10.csv", index=False)
print("chosen settings:")
print(best.groupby(level="dataset").value_counts().to_string())
pd.concat([paper, tuned]).groupby(["dataset", "method"])[["C-index", "IBS", "k"]].agg(["mean", "std"]).round(3)

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.


[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:  5.9min


[Parallel(n_jobs=6)]: Done  90 out of  90 | elapsed: 10.4min finished
[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.


[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:  6.2min


[Parallel(n_jobs=6)]: Done 150 tasks      | elapsed: 26.9min


[Parallel(n_jobs=6)]: Done 353 tasks      | elapsed: 84.2min


[Parallel(n_jobs=6)]: Done 540 out of 540 | elapsed: 89.4min finished


chosen settings:
dataset  method                      
FLChain  K=2, d=8, log-normal            2
         K=3, d=8, Weibull, learned k    2
         K=3, d=8, log-normal            2
         K=2, d=4, Weibull, k = 1        1
         K=3, d=4, Weibull, learned k    1
         K=3, d=4, log-normal            1
         K=4, d=4, log-normal            1
GBSG2    K=4, d=8, Weibull, learned k    2
         K=2, d=4, Weibull, k = 1        1
         K=2, d=4, Weibull, learned k    1
         K=2, d=8, Weibull, k = 1        1
         K=2, d=8, log-normal            1
         K=3, d=4, Weibull, k = 1        1
         K=3, d=8, Weibull, learned k    1
         K=4, d=4, Weibull, k = 1        1
         K=4, d=8, log-normal            1
WHAS500  K=2, d=4, Weibull, k = 1        1
         K=2, d=4, Weibull, learned k    1
         K=2, d=4, log-normal            1
         K=2, d=8, Weibull, k = 1        1
         K=2, d=8, log-normal            1
         K=3, d=4, Weibull, k = 1        1

C-index           IBS             k       
                                 mean    std   mean    std   mean    std
dataset method                                                          
FLChain VaDeSC (paper config)   0.794  0.011  0.109  0.002  1.000  0.000
        VaDeSC (tuned)          0.796  0.007  0.108  0.002  1.187  0.114
GBSG2   VaDeSC (paper config)   0.666  0.032  0.188  0.006  1.000  0.000
        VaDeSC (tuned)          0.665  0.036  0.187  0.010  1.282  0.291
WHAS500 VaDeSC (paper config)   0.752  0.025  0.178  0.020  1.000  0.000
        VaDeSC (tuned)          0.750  0.026  0.174  0.018  0.859  0.167

## 3. Comparison

Mean C-index over the 10 splits, and the tests: tuned VaDeSC against the paper settings and against each classical model.

In [4]:
allr = pd.concat([cls, paper, tuned])
per = allr.groupby(["dataset", "method", "split"])["C-index"].mean()
print(per.groupby(["dataset", "method"]).agg(["mean", "std"]).unstack("dataset").round(3).to_string())
tests = []
for ds in LOAD:
    ps = per.loc[ds].unstack("method")
    comps = [("VaDeSC (tuned)", "VaDeSC (paper config)")] + [(m, "VaDeSC (tuned)") for m in ["Cox PH", "RSF", "Weibull AFT", "DSM"]]
    tests += [{"dataset": ds, **split_test(ps, a, b, len(comps))} for a, b in comps]
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(tests).round(4)

                         mean                    std               
dataset               FLChain  GBSG2 WHAS500 FLChain  GBSG2 WHAS500
method                                                             
Cox PH                  0.794  0.645   0.761   0.006  0.022   0.031
DSM                     0.795  0.658   0.757   0.007  0.024   0.024
RSF                     0.789  0.677   0.759   0.007  0.029   0.023
VaDeSC (paper config)   0.794  0.666   0.752   0.008  0.032   0.025
VaDeSC (tuned)          0.796  0.665   0.750   0.007  0.036   0.024
Weibull AFT             0.795  0.645   0.761   0.006  0.022   0.031


,dataset,comparison,mean diff,95% CI,higher in,p-value,p (Bonferroni)
0,FLChain,VaDeSC (tuned) minus VaDeSC (paper config),0.0019,"[-0.002, +0.006]",5/10 splits,0.3163,1.0000
1,FLChain,Cox PH minus VaDeSC (tuned),-0.0017,"[-0.004, +0.001]",1/10 splits,0.1950,0.9752
2,FLChain,RSF minus VaDeSC (tuned),-0.0074,"[-0.011, -0.004]",0/10 splits,0.0012,0.0059
3,FLChain,Weibull AFT minus VaDeSC (tuned),-0.0012,"[-0.004, +0.001]",2/10 splits,0.3278,1.0000
4,FLChain,DSM minus VaDeSC (tuned),-0.0015,"[-0.007, +0.003]",6/10 splits,0.5017,1.0000
5,GBSG2,VaDeSC (tuned) minus VaDeSC (paper config),-0.0009,"[-0.008, +0.007]",5/10 splits,0.8021,1.0000
6,GBSG2,Cox PH minus VaDeSC (tuned),-0.0204,"[-0.043, +0.003]",3/10 splits,0.0773,0.3867
7,GBSG2,RSF minus VaDeSC (tuned),0.0122,"[-0.007, +0.031]",8/10 splits,0.1806,0.9029
8,GBSG2,Weibull AFT minus VaDeSC (tuned),-0.0204,"[-0.043, +0.003]",3/10 splits,0.0758,0.3789
9,GBSG2,DSM minus VaDeSC (tuned),-0.0074,"[-0.032, +0.017]",4/10 splits,0.5164,1.0000


## Notes

- On FLChain all models are almost the same (0.789-0.796).
- On GBSG2 RSF is best (0.677) and Cox and Weibull AFT are lowest (0.645), with VaDeSC in between (0.666). On WHAS500 VaDeSC is slightly lowest (0.750-0.752 vs 0.757-0.761).
- After the Bonferroni correction only one difference is significant: on FLChain RSF is a bit worse than the tuned VaDeSC (-0.007, p = 0.006). On GBSG2 and WHAS500 the 95% CIs are about +-0.02, so we cannot rule out differences of that size.
- Tuning did not help: the tuned VaDeSC is within 0.002 of the paper settings on every dataset. The chosen setting changed a lot between splits (no setting was chosen more than 2 times out of 10), so the differences in validation C-index are probably mostly noise (the validation sets are small).
- So on these datasets VaDeSC predicts about as well as the classical models, unlike on SUPPORT where it is clearly worse (notebook 16).